# Team I3 — Baseline Models

Runs the two non-deep-learning baselines separately from the neural-network notebook:
1. Mean-life predictor (training-target mean only).
2. Elastic-Net regression using the preprocessed ΔQ(V) vectors.

Inputs: `data/processed/normalized_{10,30,100}_{train,val,test}.npz`, containing `delta`, `y`, and `ids`.

Outputs are saved under `results/baselines/`. The Elastic-Net artifacts are also used by the optional CNN–Elastic-Net fusion section in `all_models_training.ipynb`.

In [4]:
from pathlib import Path
import warnings, joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")

In [5]:
PROJECT_ROOT = Path(r"C:\Users\lokan\Document\DL\ENDSEM")
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
RESULT_ROOT = PROJECT_ROOT / "results" / "baselines"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
HORIZONS = [10, 30, 100]
SPLITS = ["train", "val", "test"]
SEED = 22

In [6]:
def load_split(h, split):
    path = PROCESSED_ROOT / f"normalized_{h}_{split}.npz"
    if not path.exists():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=True) as z:
        return {"X": np.asarray(z["delta"], dtype=float),
                "y": np.asarray(z["y"], dtype=float).reshape(-1),
                "ids": np.asarray(z["ids"]).astype(str)}

DATA = {h: {s: load_split(h, s) for s in SPLITS} for h in HORIZONS}
for h in HORIZONS:
    for s in SPLITS:
        d = DATA[h][s]
        assert len(d["X"]) == len(d["y"]) == len(d["ids"])
        assert np.isfinite(d["X"]).all() and np.isfinite(d["y"]).all()
        assert (d["y"] > 0).all()
        print(h, s, d["X"].shape, d["y"].shape)

10 train (75, 1000) (75,)
10 val (24, 1000) (24,)
10 test (29, 1000) (29,)
30 train (75, 1000) (75,)
30 val (24, 1000) (24,)
30 test (29, 1000) (29,)
100 train (75, 1000) (75,)
100 val (24, 1000) (24,)
100 test (29, 1000) (29,)


## Evaluation utilities

All metrics are in cycle-life units. MAPE is reported as a percentage.

In [7]:
def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"MAE": mean_absolute_error(y, p),
            "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "MAPE_percent": float(np.mean(np.abs((y-p)/y))*100),
            "R2": r2_score(y, p)}

def save_predictions(model, h, split, ids, y, p):
    pd.DataFrame({"sample_id": ids, "y_true": y, "y_pred": p,
                  "residual": y-p}).to_csv(
        RESULT_ROOT / f"{model}_{h}_{split}_predictions.csv", index=False)

## 1. Mean-life predictor

For each horizon, calculate the mean target from the training partition only. Apply that constant prediction to each partition. Do not use validation or test targets to calculate the mean.

In [8]:
mean_rows = []
for h in HORIZONS:
    train_mean = float(DATA[h]["train"]["y"].mean())
    print(f"{h}-cycle training mean: {train_mean:.4f}")
    for split in SPLITS:
        d = DATA[h][split]
        p = np.full(len(d["y"]), train_mean)
        mean_rows.append({"model": "Mean-life", "horizon_cycles": h,
                          "split": split, "training_target_mean": train_mean,
                          "n_samples": len(d["y"]), **metrics(d["y"], p)})
        save_predictions("mean_life", h, split, d["ids"], d["y"], p)

mean_df = pd.DataFrame(mean_rows)
mean_df.to_csv(RESULT_ROOT / "mean_life_metrics.csv", index=False)
display(mean_df)

10-cycle training mean: 824.6000
30-cycle training mean: 824.6000
100-cycle training mean: 824.6000


,model,horizon_cycles,split,training_target_mean,n_samples,MAE,RMSE,MAPE_percent,R2
0,Mean-life,10,train,824.6,75,291.749333,385.908383,45.196135,0.000000
1,Mean-life,10,val,824.6,24,315.766667,434.753965,39.096353,-0.006643
2,Mean-life,10,test,824.6,29,250.689655,338.711689,37.401715,-0.028978
3,Mean-life,30,train,824.6,75,291.749333,385.908383,45.196135,0.000000
4,Mean-life,30,val,824.6,24,315.766667,434.753965,39.096353,-0.006643
5,Mean-life,30,test,824.6,29,250.689655,338.711689,37.401715,-0.028978
6,Mean-life,100,train,824.6,75,291.749333,385.908383,45.196135,0.000000
7,Mean-life,100,val,824.6,24,315.766667,434.753965,39.096353,-0.006643
8,Mean-life,100,test,824.6,29,250.689655,338.711689,37.401715,-0.028978


## 2. Elastic-Net regression

Candidate hyperparameters are selected using validation RMSE in cycle units. The selected model is then refitted on train + validation and evaluated on the held-out test split. The target is fitted in log space and transformed back to cycles for reporting.

In [9]:
ALPHAS = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]
L1_RATIOS = [0.1, 0.5, 0.9, 1.0]
enet_rows, search_rows = [], []

for h in HORIZONS:
    tr, va, te = (DATA[h][s] for s in SPLITS)
    best_cfg, best_rmse = None, np.inf

    for alpha in ALPHAS:
        for ratio in L1_RATIOS:
            candidate = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", ElasticNet(alpha=alpha, l1_ratio=ratio,
                                     max_iter=100000, random_state=SEED))
            ])
            candidate.fit(tr["X"], np.log(tr["y"]))
            p_val = np.exp(np.clip(candidate.predict(va["X"]), -20, 20))
            m = metrics(va["y"], p_val)
            search_rows.append({"horizon_cycles": h, "alpha": alpha,
                                "l1_ratio": ratio, **m})
            if m["RMSE"] < best_rmse:
                best_rmse, best_cfg = m["RMSE"], (alpha, ratio)

    alpha, ratio = best_cfg
    X_fit = np.concatenate([tr["X"], va["X"]])
    y_fit = np.concatenate([tr["y"], va["y"]])
    final_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=alpha, l1_ratio=ratio,
                             max_iter=100000, random_state=SEED))
    ])
    final_model.fit(X_fit, np.log(y_fit))
    joblib.dump(final_model, RESULT_ROOT / f"elastic_net_{h}.joblib")

    for split, d in zip(SPLITS, [tr, va, te]):
        p = np.exp(np.clip(final_model.predict(d["X"]), -20, 20))
        enet_rows.append({"model": "Elastic-Net", "horizon_cycles": h,
                          "split": split, "alpha": alpha, "l1_ratio": ratio,
                          "n_samples": len(d["y"]), **metrics(d["y"], p)})
        save_predictions("elastic_net", h, split, d["ids"], d["y"], p)

    print(f"Horizon {h}: alpha={alpha}, l1_ratio={ratio}, "
          f"best validation RMSE={best_rmse:.3f}")

Horizon 10: alpha=0.01, l1_ratio=0.5, best validation RMSE=376.588
Horizon 30: alpha=0.01, l1_ratio=0.5, best validation RMSE=295.429
Horizon 100: alpha=0.001, l1_ratio=1.0, best validation RMSE=165.493


In [10]:
enet_df = pd.DataFrame(enet_rows)
search_df = pd.DataFrame(search_rows)
enet_df.to_csv(RESULT_ROOT / "elastic_net_metrics.csv", index=False)
search_df.to_csv(RESULT_ROOT / "elastic_net_validation_search.csv", index=False)

test_comparison = pd.concat([
    mean_df[mean_df["split"] == "test"],
    enet_df[enet_df["split"] == "test"]
], ignore_index=True)
test_comparison.to_csv(RESULT_ROOT / "baseline_test_comparison.csv", index=False)

display(enet_df)
display(test_comparison[["model", "horizon_cycles", "n_samples",
                         "MAE", "RMSE", "MAPE_percent", "R2"]])
print("Baseline outputs saved to:", RESULT_ROOT)

,model,horizon_cycles,split,alpha,l1_ratio,n_samples,MAE,RMSE,MAPE_percent,R2
0,Elastic-Net,10,train,0.010,0.5,75,190.720258,305.731564,21.575885,0.372358
1,Elastic-Net,10,val,0.010,0.5,24,234.611755,334.665723,26.248260,0.403500
2,Elastic-Net,10,test,0.010,0.5,29,194.404391,262.878898,26.678651,0.380192
3,Elastic-Net,30,train,0.010,0.5,75,158.527800,240.502740,19.523375,0.611607
4,Elastic-Net,30,val,0.010,0.5,24,169.198396,257.164121,18.847226,0.647784
5,Elastic-Net,30,test,0.010,0.5,29,154.817577,210.255215,20.901275,0.603503
6,Elastic-Net,100,train,0.001,1.0,75,90.979219,134.328047,11.001136,0.878838
7,Elastic-Net,100,val,0.001,1.0,24,95.335200,132.921776,10.414042,0.905902
8,Elastic-Net,100,test,0.001,1.0,29,82.578431,105.610507,11.537367,0.899963


,model,horizon_cycles,n_samples,MAE,RMSE,MAPE_percent,R2
0,Mean-life,10,29,250.689655,338.711689,37.401715,-0.028978
1,Mean-life,30,29,250.689655,338.711689,37.401715,-0.028978
2,Mean-life,100,29,250.689655,338.711689,37.401715,-0.028978
3,Elastic-Net,10,29,194.404391,262.878898,26.678651,0.380192
4,Elastic-Net,30,29,154.817577,210.255215,20.901275,0.603503
5,Elastic-Net,100,29,82.578431,105.610507,11.537367,0.899963


Baseline outputs saved to: C:\Users\lokan\Document\DL\ENDSEM\results\baselines
